# Lab C · Change the question. Explain the difference.

**Question:** Do the same counties stand out if we rank by **average paid amount** instead of **claim count**?

**Measure:** mean of `AMOUNT_PAID_ON_BUILDING_CLAIM` over records where it is present. Missing payments are **not** treated as zero. The denominator and the missing count are shown so you can judge each mean.

**Data:** FEMA NFIP claims for Hurricane Sandy in New York and New Jersey, plus county boundaries, from the free *Snowflake Public Data (Free)* Marketplace listing.

> This is descriptive history for one event. It is not a calibrated flood risk model and says nothing about any individual property.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.use_warehouse("SDSC_WORKSHOP_WH")

listing_db = "SNOWFLAKE_PUBLIC_DATA_FREE"   # <-- change if you renamed it
claims_tbl = f"{listing_db}.PUBLIC_DATA_FREE.FEMA_NATIONAL_FLOOD_INSURANCE_PROGRAM_CLAIM_INDEX"
geo_index  = f"{listing_db}.PUBLIC_DATA_FREE.GEOGRAPHY_INDEX"
geo_chars  = f"{listing_db}.PUBLIC_DATA_FREE.GEOGRAPHY_CHARACTERISTICS"

min_claims = 500   # <-- TRY: change to 5000, then to 100, and rerun all cells

In [ ]:
%%sql -r county_ranks
WITH county_stats AS (
  SELECT county_geo_id,
         COUNT(*)                                        AS claim_records,
         COUNT(amount_paid_on_building_claim)            AS paid_denominator,
         COUNT(*) - COUNT(amount_paid_on_building_claim) AS paid_missing,
         AVG(amount_paid_on_building_claim)              AS mean_bldg_paid_usd
  FROM {{claims_tbl}}
  WHERE flood_event   = 'Hurricane Sandy'
    AND state_geo_id IN ('geoId/36', 'geoId/34')
    AND county_geo_id IS NOT NULL
  GROUP BY county_geo_id
  HAVING COUNT(*) >= {{min_claims}}
)
SELECT c.county_geo_id,
       g.geo_name                                     AS county,
       claim_records,
       paid_denominator,
       paid_missing,
       ROUND(mean_bldg_paid_usd)                      AS mean_bldg_paid_usd,
       RANK() OVER (ORDER BY claim_records DESC)      AS rank_by_count,
       RANK() OVER (ORDER BY mean_bldg_paid_usd DESC) AS rank_by_mean_paid
FROM county_stats c
JOIN {{geo_index}} g ON g.geo_id = c.county_geo_id
ORDER BY rank_by_count

In [ ]:
import pandas as pd
import altair as alt
alt.renderers.enable("mimetype")

ranks = county_ranks.copy() if isinstance(county_ranks, pd.DataFrame) else county_ranks.to_pandas()

# Slope chart: each line is a county moving from its count rank to its mean-paid rank.
long = ranks.melt(id_vars=["COUNTY"], value_vars=["RANK_BY_COUNT", "RANK_BY_MEAN_PAID"],
                  var_name="RANKING", value_name="RANK")
long["RANKING"] = long["RANKING"].map({"RANK_BY_COUNT": "By claim count", "RANK_BY_MEAN_PAID": "By mean paid"})

base = alt.Chart(long).encode(
    x=alt.X("RANKING:N", title=None, sort=["By claim count", "By mean paid"], axis=alt.Axis(labelAngle=0)),
    y=alt.Y("RANK:Q", scale=alt.Scale(reverse=True), title="Rank (1 = top)"),
    color=alt.Color("COUNTY:N", legend=None),
)
slope = (base.mark_line(point=True)
         + base.mark_text(align="left", dx=6).encode(text="COUNTY:N").transform_filter(alt.datum.RANKING == "By mean paid")
        ).properties(width=320, height=380, title="Same counties, different question")

# Bars: claim volume vs mean payout side by side.
bars_count = alt.Chart(ranks).mark_bar(color="#29B5E8").encode(
    x=alt.X("CLAIM_RECORDS:Q", title="Claim records"),
    y=alt.Y("COUNTY:N", sort="-x", title=None),
    tooltip=["COUNTY", "CLAIM_RECORDS", "PAID_DENOMINATOR", "PAID_MISSING"],
).properties(width=260, height=380, title="Claim count")
bars_mean = alt.Chart(ranks).mark_bar(color="#FF9F36").encode(
    x=alt.X("MEAN_BLDG_PAID_USD:Q", title="Mean building paid (USD)"),
    y=alt.Y("COUNTY:N", sort="-x", title=None),
    tooltip=["COUNTY", "MEAN_BLDG_PAID_USD", "PAID_DENOMINATOR", "PAID_MISSING"],
).properties(width=260, height=380, title="Mean building payout")

slope | bars_count | bars_mean

In [ ]:
%%sql -r county_shapes
-- Real county boundaries (WKT text -> GEOGRAPHY), simplified to ~200 m for fast drawing.
SELECT r.county_geo_id,
       ST_ASGEOJSON(ST_SIMPLIFY(TO_GEOGRAPHY(gc.value), 200))::STRING AS geojson
FROM {{county_ranks}} r
JOIN {{geo_chars}} gc
  ON gc.geo_id = r.county_geo_id
 AND gc.relationship_type = 'coordinates_wkt'

In [ ]:
import json

shapes = county_shapes.copy() if isinstance(county_shapes, pd.DataFrame) else county_shapes.to_pandas()
geo = shapes.merge(ranks, on="COUNTY_GEO_ID")

def for_d3(geom):
    # Snowflake emits counter-clockwise rings; Altair/d3 wants clockwise, so reverse them.
    if geom["type"] == "Polygon":
        geom["coordinates"] = [ring[::-1] for ring in geom["coordinates"]]
    elif geom["type"] == "MultiPolygon":
        geom["coordinates"] = [[ring[::-1] for ring in poly] for poly in geom["coordinates"]]
    return geom

features = [
    {"type": "Feature", "geometry": for_d3(json.loads(row.GEOJSON)),
     "properties": {"county": row.COUNTY, "claims": int(row.CLAIM_RECORDS),
                    "mean_paid": float(row.MEAN_BLDG_PAID_USD), "missing": int(row.PAID_MISSING),
                    "rank_count": int(row.RANK_BY_COUNT), "rank_mean": int(row.RANK_BY_MEAN_PAID)}}
    for row in geo.itertuples()
]
data = alt.Data(values=features)
tooltip = [alt.Tooltip("properties.county:N", title="County"),
           alt.Tooltip("properties.claims:Q", title="Claims", format=","),
           alt.Tooltip("properties.mean_paid:Q", title="Mean paid", format="$,.0f"),
           alt.Tooltip("properties.missing:Q", title="Missing paid"),
           alt.Tooltip("properties.rank_count:Q", title="Rank by count"),
           alt.Tooltip("properties.rank_mean:Q", title="Rank by mean")]

def county_map(field, title, scheme):
    return alt.Chart(data).mark_geoshape(stroke="white", strokeWidth=0.7).encode(
        color=alt.Color(f"properties.{field}:Q", title=title, scale=alt.Scale(scheme=scheme)),
        tooltip=tooltip,
    ).project(type="mercator").properties(width=380, height=440, title=title)

(county_map("claims", "Claim records", "blues")
 | county_map("mean_paid", "Mean building paid (USD)", "oranges")
).resolve_scale(color="independent")

## What you should see (min_claims = 500, snapshot 2026-10-11; the listing refreshes, so expect small drift)

| County | Claims | Mean paid | Rank by count | Rank by mean |
|---|---:|---:|:-:|:-:|
| Ocean County | 37,459 | $59,274 | 1 | 9 |
| Nassau County | 26,964 | $67,115 | 2 | 6 |
| Atlantic County | 11,344 | $35,224 | 4 | 12 |
| Brooklyn | 3,561 | $81,335 | 9 | 2 |
| Manhattan | 668 | $236,483 | 13 | 1 (223 of 668 missing) |

Full 14-row table: `labs/03_change_the_question.sql` or `labs/EXPECTED_RESULTS.md`.

**The story flips.** On the left map the Jersey Shore and Long Island are darkest. On the right map Manhattan is. Manhattan has relatively few claim records but the highest mean building payout, and Ocean County has the most records but ranks 9th on mean payout. Neither ranking is "right". They answer different questions.

**Your change:** set `min_claims = 5000` in the Parameters cell and rerun all cells. Manhattan and Brooklyn drop off the map entirely. The threshold you pick changes which story you can tell.

**Interpret:** a different metric changes the answer. Before an AI explains this table, you should already know what it should say.